# 03 · Indexing and broadcasting real data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb)

*Part III · exercise · 15 min*

> 🇪🇸 **Indexación y broadcasting con datos reales** — Seleccionar la columna correcta de datos reales de tumores y encontrar píxeles de varianza cero.

Select the right column of real tumour data, then meet zero-variance pixels.

## What you will be able to do

- Select a named column of real data by name, never by a hard-coded number.
- Combine fancy indexing and boolean indexing to pull out sub-tables in one operation.
- Standardize a data matrix with broadcasting.
- Recognise a zero-variance column, and know why real images contain them.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_digits

bc = load_breast_cancer()
X, y = bc.data, bc.target          # (569, 30); y: 0 = malignant, 1 = benign
names = list(bc.feature_names)
print(X.shape, len(names))

## Why this matters

> 🇪🇸 Elegir la columna equivocada no da error: devuelve otra medida real, y el
> análisis continúa y da una respuesta segura y equivocada.

The `breast_cancer` data holds 30 real measurements of tumour cell nuclei for
569 real patients. Selecting the wrong column does not produce an error — it
returns a *different real measurement*, and your analysis continues and gives a
confident, wrong answer.

In research this produces results nobody can reproduce. In a clinical tool it
produces a wrong recommendation about a real person.

**In tech**, the identical operation runs on a `(users, items)` matrix to pull
one user's history before making a recommendation.

## Exercise 1 — indexing by name

> 🇪🇸 Indexación por nombre, nunca por número fijo.

In [ ]:
# TODO 1: Print X.shape. Say out loud what each axis means.

# TODO 2: Extract the column "mean radius" for all patients -> shape (569,).
#         Find its position with names.index(...). Do not hard-code a number.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(X.shape)                 # (569, 30)  patients x measurements

i = names.index("mean radius")
radius = X[:, i]               # book notation A_{:,j}
print(i, radius.shape)         # 0 (569,)

# names.index() rather than 0 because the column order is not yours to assume.
# If the dataset is ever reordered, the hard-coded version keeps running and
# keeps being wrong.

## Exercise 2 — fancy and boolean indexing

> 🇪🇸 Indexación avanzada y booleana.

In [ ]:
# TODO 3: Find the 5 patients with the LARGEST mean radius, then extract their
#         full 30-measurement profiles as one (5, 30) array, in ONE operation.

# TODO 4: Using boolean indexing, compare mean radius for malignant (y == 0)
#         against benign (y == 1) patients. Is there a real difference?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
top5 = np.argsort(radius)[-5:]
profiles = X[top5, :]                               # (5, 30)
print(profiles.shape)

print(radius[y == 0].mean(), radius[y == 1].mean()) # 17.5 vs 12.1

# A real result: MALIGNANT TUMOURS REALLY DO HAVE A LARGER MEAN RADIUS,
# 17.5 against 12.1. Random data would never have shown you that.

## Broadcasting, on real images

> 🇪🇸 Broadcasting sobre imágenes reales.

Broadcasting stretches a smaller array across a larger one without copying it.
Standardizing a data matrix — subtract the mean of each column, divide by its
standard deviation — is the operation you will do most often.

Run TODO 6 and **look at the result before continuing**. Something is wrong with
it, and finding out what is the point of this block.

In [ ]:
images = load_digits().images          # (1797, 8, 8)
D = images.reshape(len(images), -1)    # (1797, 64)
print(D.shape)

## Exercise 3 — standardize, then find the trap

> 🇪🇸 Estandariza y encuentra el problema.

In [ ]:
# TODO 5: Compute the mean and std of each of the 64 pixels across all images.

# TODO 6: Standardize with broadcasting: (D - mean) / std.
#         RUN IT AND LOOK AT THE RESULT before continuing.

# TODO 7: You will find NaN. How many pixels have std == 0, and why would a real
#         handwritten digit image contain such pixels? Fix it, then verify no NaN.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
mean, std = D.mean(axis=0), D.std(axis=0)
print(mean.shape, std.shape)                        # (64,) (64,)

Z_bad = (D - mean) / std
print(np.isnan(Z_bad).any())                        # True

print((std == 0).sum())                             # 3
Z = (D - mean) / np.where(std == 0, 1.0, std)
print(np.isnan(Z).any())                            # False

# THREE PIXELS ARE ALWAYS DARK in all 1797 digit images: they sit in corners
# where nobody writes. Their standard deviation is exactly zero, so dividing
# produces NaN. np.where leaves those columns as plain centred zeros, which is
# the honest thing to do with a feature that carries no information.

## What just happened

Two real results, neither of which random data could have produced:

1. **Malignant tumours really do have a larger mean radius** — 17.5 against 12.1.
2. **Three pixels are always dark** in all 1797 digit images, so their standard
   deviation is exactly zero and dividing by it produces `NaN`.

The second is the one to remember. A zero-variance feature is not a bug in your
code — it is a fact about your data, and you have to decide what to do about it.
Silently propagating `NaN` into a model is the one option that is always wrong.

The Kahoot below asks you about exactly this.

---

## Done with this section

Next up: **04 · Reshape and transpose real images** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)